# Exploration de l'API INSEE BDM

Ce notebook sert à valider la structure des réponses de la Banque de Données Macroéconomiques de l'INSEE pour notre PoC.

In [6]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

print("Environnement initialisé.")

Environnement initialisé.


## 1. Requête sur l'IPC (Inflation)
Identifiant de famille (Dataflow) : `IPC-2015`

L'INSEE utilise le standard SDMX. Le point d'entrée pour récupérer la structure est :
`https://bdm.insee.fr/series/sdmx/datastructure/FR1/IPC-2015`

In [7]:
url = "https://bdm.insee.fr/series/sdmx/datastructure/FR1/IPC-2015"
response = requests.get(url)

if response.status_code == 200:
    print("Requête réussie ! Extraction des dimensions en cours...\n")
    
    # Parsing du XML avec ElementTree
    root = ET.fromstring(response.content)
    
    # Le SDMX utilise des namespaces (espaces de noms) stricts qu'il faut déclarer
    ns = {
        'mes': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message',
        'str': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure',
        'com': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common'
    }
    
    # On cherche la liste des dimensions dans la structure du message
    dimensions = root.findall('.//str:DimensionList/str:Dimension', ns)
    
    print("=== DIMENSIONS OBLIGATOIRES POUR LA FAMILLE IPC-2015 ===")
    for dim in dimensions:
        dim_id = dim.get('id')
        print(f"- {dim_id}")
        
else:
    print(f"Erreur {response.status_code}")

Requête réussie ! Extraction des dimensions en cours...

=== DIMENSIONS OBLIGATOIRES POUR LA FAMILLE IPC-2015 ===
- FREQ
- INDICATEUR
- FORME-VENTE
- COICOP2016
- PRIX_CONSO
- NATURE
- MENAGES_IPC
- REF_AREA
- UNIT_MEASURE
- CORRECTION
- BASIND
- SERIE_ARRETEE


## 2. Exploration des autres domaines (Chômage et Démographie)
On répète l'opération pour les autres Dataflows.

In [8]:
dataflows = ['CHOMAGE-TRIM-NATIONAL', 'NAISSANCES-FECONDITE']
for df in dataflows:
    url = f"https://bdm.insee.fr/series/sdmx/datastructure/FR1/{df}"
    res = requests.get(url)
    if res.status_code == 200:
        root = ET.fromstring(res.content)
        dims = root.findall('.//str:DimensionList/str:Dimension', ns)
        print(f"\n=== DIMENSIONS POUR {df} ===")
        for dim in dims:
            print(f"- {dim.get('id')}")
    else:
        print(f"Erreur sur {df}")



=== DIMENSIONS POUR CHOMAGE-TRIM-NATIONAL ===
- FREQ
- INDICATEUR
- NATURE
- REF_AREA
- SEXE
- AGE
- UNIT_MEASURE
- CORRECTION
- SERIE_ARRETEE

=== DIMENSIONS POUR NAISSANCES-FECONDITE ===
- FREQ
- INDICATEUR
- NATURE
- DEMOGRAPHIE
- REF_AREA
- AGE
- UNIT_MEASURE
- CORRECTION
- SERIE_ARRETEE


## 3. Le Test de Vérité (Les Observations)
Récupérons la valeur de l'inflation d'Août 2022 (MVP-002) : *"Les prix à la consommation ont augmenté de 5,8 % entre août 2021 et août 2022."*

La façon la plus simple de récupérer les valeurs dans la BDM est d'utiliser l'identifiant unique de la série (`idbank`) et de télécharger le CSV officiel.
L'idbank pour *"IPC - Ensemble des ménages - France - Ensemble - Glissement annuel"* est **1763852**.

In [9]:
# Récupération de la donnée via l'API SDMX (format XML)
idbank = "001763852"
url_data = f"https://bdm.insee.fr/series/sdmx/data/SERIES_BDM/{idbank}"

print(f"Téléchargement des données depuis {url_data}...")
res = requests.get(url_data)

if res.status_code == 200:
    root = ET.fromstring(res.content)
    
    # Extraction des observations (Période et Valeur)
    observations = []
    # Le tag Obs contient les attributs TIME_PERIOD et OBS_VALUE
    for obs in root.iter():
        if 'Obs' in obs.tag:
            observations.append({
                'Periode': obs.get('TIME_PERIOD'),
                'Valeur': float(obs.get('OBS_VALUE'))
            })
            
    df_clean = pd.DataFrame(observations)
    
    # Filtre sur Août 2022
    val_aout_2022 = df_clean[df_clean['Periode'] == '2022-08']
    print("\nRésultat pour Août 2022 :\n")
    print(val_aout_2022.to_string(index=False))
else:
    print("Erreur :", res.status_code)


Téléchargement des données depuis https://bdm.insee.fr/series/sdmx/data/SERIES_BDM/001763852...

Résultat pour Août 2022 :

Periode  Valeur
2022-08  112.63


## 4. Validation exhaustive de la Cartographie (Lot 1 complet)

Grâce au script `fetch_dataflows.py`, nous avons interrogé l'API INSEE pour valider l'existence et la structure de l'ensemble des Dataflows nécessaires à nos 30 affirmations. 
Vérifions le résultat de cet export :

In [10]:
import pandas as pd

df_carto = pd.read_csv("../data/selected_dataflows.csv")

# On affiche uniquement les Dataflows du MVP
mvp_dataflows = df_carto[df_carto['tier'] == 'MVP']
display(mvp_dataflows)

print(f"\n{len(mvp_dataflows)} Dataflows MVP ont été validés avec succès dans l'API BDM.")

,dataflow_id,title,tier,exists,dimension_count,dimensions,error
0,IPC-2025,Indices des prix à la consommation,MVP,True,13,TIME_PERIOD | FREQ | INDICATEUR | FORME-VENTE ...,NaN
1,IPCH-2025,Indices des prix à la consommation harmonisés,MVP,True,10,TIME_PERIOD | FREQ | INDICATEUR | COICOP2018 |...,NaN
2,CHOMAGE-TRIM-NATIONAL,"Chômage, taux de chômage par sexe et âge (sens...",MVP,True,10,TIME_PERIOD | FREQ | INDICATEUR | NATURE | REF...,NaN
3,EMPLOI-BIT-TRIM,"Activité, emploi, sous-emploi au sens du BIT e...",MVP,True,10,TIME_PERIOD | FREQ | INDICATEUR | NATURE | REF...,NaN
4,EMPLOI-SALARIE-TRIM-NATIONAL,Estimations d'emploi salarié par secteur d'act...,MVP,True,10,TIME_PERIOD | FREQ | INDICATEUR | NAF2 | NAF2_...,NaN
5,DEMANDES-EMPLOIS-NATIONALES,Demandeurs d'emploi inscrits à Pôle Emploi,MVP,True,14,TIME_PERIOD | FREQ | INDICATEUR | NATURE | MET...,NaN
6,NAISSANCES-FECONDITE,Naissances et fécondité,MVP,True,10,TIME_PERIOD | FREQ | INDICATEUR | NATURE | DEM...,NaN
7,POPULATION-STRUCTURE,Population et structure de la population,MVP,True,11,TIME_PERIOD | FREQ | INDICATEUR | NATURE | DEM...,NaN
8,DECES-MORTALITE,Décès et mortalité,MVP,True,11,TIME_PERIOD | FREQ | INDICATEUR | NATURE | DEM...,NaN
9,CREATIONS-ENTREPRISES-METHODE-2022,Créations d'entreprises - Méthode 2022,MVP,True,11,TIME_PERIOD | FREQ | EUROSTAT_CREATIONS_ENTREP...,NaN



10 Dataflows MVP ont été validés avec succès dans l'API BDM.


In [1]:
import pandas as pd

# 5. Mapping exhaustif des 30 affirmations MVP
mapping_data = [
    {"id": "MVP-001", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "Nécessite pondération générale vs 38 produits"},
    {"id": "MVP-002", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "Variation annuelle classique (GLISSEMENT)"},
    {"id": "MVP-003", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "COICOP=01 (Alimentation)"},
    {"id": "MVP-004", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "COICOP=04 (Energie)"},
    {"id": "MVP-005", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "Comparaison de deux glissements"},
    {"id": "MVP-006", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "NATURE=MOYENNE"},
    {"id": "MVP-007", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPC-2025", "remarque": "Glissement annuel"},
    {"id": "MVP-008", "theme": "Inflation", "verifiable_bdm": True, "dataflow": "IPCH-2025", "remarque": "Comparaison France / Zone Euro (IPCH)"},
    
    {"id": "MVP-009", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "CHOMAGE-TRIM-NATIONAL", "remarque": "Différence temporelle"},
    {"id": "MVP-010", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "CHOMAGE-TRIM-NATIONAL", "remarque": "Minimum historique"},
    {"id": "MVP-011", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "CHOMAGE-TRIM-NATIONAL", "remarque": "AGE=15-24"},
    {"id": "MVP-012", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "DEMANDES-EMPLOIS-NATIONALES", "remarque": "Nécessite DARES en contraste"},
    {"id": "MVP-013", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "CHOMAGE-TRIM-NATIONAL", "remarque": "Tendance sur 5 ans"},
    {"id": "MVP-014", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "EMPLOI-SALARIE-TRIM-NATIONAL", "remarque": "Emplois nets créés"},
    {"id": "MVP-015", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "EMPLOI-SALARIE-TRIM-NATIONAL", "remarque": "Record historique"},
    {"id": "MVP-016", "theme": "Emploi", "verifiable_bdm": True, "dataflow": "EMPLOI-BIT-TRIM", "remarque": "Taux d'activité"},
    
    {"id": "MVP-017", "theme": "Démographie", "verifiable_bdm": True, "dataflow": "NAISSANCES-FECONDITE", "remarque": "Valeur provisoire vs définitive"},
    {"id": "MVP-018", "theme": "Démographie", "verifiable_bdm": True, "dataflow": "NAISSANCES-FECONDITE", "remarque": "Séries longues"},
    {"id": "MVP-019", "theme": "Démographie", "verifiable_bdm": True, "dataflow": "NAISSANCES-FECONDITE", "remarque": "Différence annuelle"},
    {"id": "MVP-020", "theme": "Démographie", "verifiable_bdm": True, "dataflow": "NAISSANCES-FECONDITE", "remarque": "ICF"},
    {"id": "MVP-021", "theme": "Démographie", "verifiable_bdm": True, "dataflow": "POPULATION-STRUCTURE", "remarque": "FM vs FE"},
    {"id": "MVP-022", "theme": "Démographie", "verifiable_bdm": False, "dataflow": "PROJECTIONS", "remarque": "Dataflow de projections manquant dans le MVP BDM actuel"},
    {"id": "MVP-023", "theme": "Démographie", "verifiable_bdm": False, "dataflow": "PROJECTIONS", "remarque": "Idem MVP-022"},
    {"id": "MVP-024", "theme": "Démographie", "verifiable_bdm": False, "dataflow": "NAISSANCES-FECONDITE", "remarque": "Dimension LIEU_NAISSANCE manquante dans la BDM"},
    
    {"id": "MVP-025", "theme": "Entreprises", "verifiable_bdm": True, "dataflow": "CREATIONS-ENTREPRISES-METHODE-2022", "remarque": "Somme annuelle"},
    {"id": "MVP-026", "theme": "Entreprises", "verifiable_bdm": True, "dataflow": "CREATIONS-ENTREPRISES-METHODE-2022", "remarque": "Variation annuelle"},
    {"id": "MVP-027", "theme": "Entreprises", "verifiable_bdm": True, "dataflow": "CREATIONS-ENTREPRISES-METHODE-2022", "remarque": "Variation mensuelle"},
    {"id": "MVP-028", "theme": "Entreprises", "verifiable_bdm": True, "dataflow": "CREATIONS-ENTREPRISES-METHODE-2022", "remarque": "Sous population (Classiques)"},
    {"id": "MVP-029", "theme": "Entreprises", "verifiable_bdm": True, "dataflow": "CREATIONS-ENTREPRISES-METHODE-2022", "remarque": "Stabilité Micro-entrepreneurs"},
    {"id": "MVP-030", "theme": "Entreprises", "verifiable_bdm": True, "dataflow": "CREATIONS-ENTREPRISES-METHODE-2022", "remarque": "Rupture de série"}
]

df_mapping = pd.DataFrame(mapping_data)
display(df_mapping.style.map(lambda x: "background-color: #ffcccc" if x is False else "", subset=["verifiable_bdm"]))

non_verifiable = df_mapping[df_mapping['verifiable_bdm'] == False]



,id,theme,verifiable_bdm,dataflow,remarque
0,MVP-001,Inflation,True,IPC-2025,Nécessite pondération générale vs 38 produits
1,MVP-002,Inflation,True,IPC-2025,Variation annuelle classique (GLISSEMENT)
2,MVP-003,Inflation,True,IPC-2025,COICOP=01 (Alimentation)
3,MVP-004,Inflation,True,IPC-2025,COICOP=04 (Energie)
4,MVP-005,Inflation,True,IPC-2025,Comparaison de deux glissements
5,MVP-006,Inflation,True,IPC-2025,NATURE=MOYENNE
6,MVP-007,Inflation,True,IPC-2025,Glissement annuel
7,MVP-008,Inflation,True,IPCH-2025,Comparaison France / Zone Euro (IPCH)
8,MVP-009,Emploi,True,CHOMAGE-TRIM-NATIONAL,Différence temporelle
9,MVP-010,Emploi,True,CHOMAGE-TRIM-NATIONAL,Minimum historique


In [2]:
print(f"\nBilan : sur les 30 affirmations, {30 - len(non_verifiable)} sont directement vérifiables avec nos Dataflows actuels.")
print(f"Les {len(non_verifiable)} autres nécessitent un ajustement (ajout de dataflow ou utilisation de Melodi) :")
for _, row in non_verifiable.iterrows():
    print(f"- {row['id']} : {row['remarque']}")


Bilan : sur les 30 affirmations, 27 sont directement vérifiables avec nos Dataflows actuels.
Les 3 autres nécessitent un ajustement (ajout de dataflow ou utilisation de Melodi) :
- MVP-022 : Dataflow de projections manquant dans le MVP BDM actuel
- MVP-023 : Idem MVP-022
- MVP-024 : Dimension LIEU_NAISSANCE manquante dans la BDM
